# OpenClaw Colab Starter

This notebook is a **public GitHub-safe starter** for running OpenClaw on **Google Colab**.

## Default configuration
- Provider: `openai`
- Model: `openai/gpt-4o-mini`

## Goals
- Remove hardcoded secrets
- Use **Colab Secrets** for sensitive values
- Keep the notebook easy to run from top to bottom
- Avoid personal backup names or user-specific paths
- Make the pairing workflow practical for the Colab environment

## Required Colab Secrets
- `OPENAI_API_KEY`
- `TELEGRAM_BOT_TOKEN`
- `ALLOWED_USER_IDS`

## Optional Colab Secrets
- `OPENCLAW_DEFAULT_MODEL`
- `OPENCLAW_MODEL_PROVIDER`
- `OPENCLAW_DATA_DIR`

> This notebook does **not** store real secrets.


## Before you start

This notebook is designed for **public GitHub upload**.

Please keep these rules:
- Do **not** hardcode API keys, bot tokens, or allowed user IDs
- Do **not** commit backup archives or runtime state files
- Clear notebook outputs before uploading
- Run cells **from top to bottom**


In [ ]:
# 1) Load Colab Secrets and set base environment variables

import os
from pathlib import Path

try:
    from google.colab import userdata
except ImportError as e:
    raise RuntimeError(
        "This notebook is intended to run on Google Colab."
    ) from e


def get_secret(name: str, default: str | None = None, required: bool = False) -> str | None:
    """Read a value from Colab Secrets and mirror it into environment variables."""
    try:
        value = userdata.get(name)
    except Exception:
        value = None

    if value in (None, ""):
        value = default

    if required and not value:
        raise ValueError(
            f"Secret '{name}' is missing. "
            "Open the left Colab panel > Secrets and add the value there."
        )

    if value:
        os.environ[name] = value
    return value


OPENAI_API_KEY = get_secret("OPENAI_API_KEY", required=True)
TELEGRAM_BOT_TOKEN = get_secret("TELEGRAM_BOT_TOKEN", required=True)
ALLOWED_USER_IDS = get_secret("ALLOWED_USER_IDS", required=True)

OPENCLAW_DEFAULT_MODEL = get_secret("OPENCLAW_DEFAULT_MODEL", default="openai/gpt-4o-mini")
OPENCLAW_MODEL_PROVIDER = get_secret("OPENCLAW_MODEL_PROVIDER", default="openai")

os.environ["OPENCLAW_DEFAULT_MODEL"] = OPENCLAW_DEFAULT_MODEL
os.environ["OPENCLAW_MODEL_PROVIDER"] = OPENCLAW_MODEL_PROVIDER
os.environ["PATH"] = os.environ.get("PATH", "") + ":/usr/local/bin"

print("Secrets loaded successfully:")
print("- OPENAI_API_KEY: OK")
print("- TELEGRAM_BOT_TOKEN: OK")
print("- ALLOWED_USER_IDS: OK")
print(f"- OPENCLAW_MODEL_PROVIDER: {OPENCLAW_MODEL_PROVIDER}")
print(f"- OPENCLAW_DEFAULT_MODEL: {OPENCLAW_DEFAULT_MODEL}")


In [ ]:
# 2) Mount Google Drive and prepare working directories

from google.colab import drive

drive.mount("/content/drive")

OPENCLAW_DATA_DIR = get_secret(
    "OPENCLAW_DATA_DIR",
    default="/content/drive/MyDrive/openclaw_data"
)

DRIVE_DATA_DIR = Path(OPENCLAW_DATA_DIR)
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_DIR = Path("/root/.openclaw")

print(f"Drive data directory: {DRIVE_DATA_DIR}")
print(f"Project directory:    {PROJECT_DIR}")


## Step 2. Install dependencies

This cell installs Node.js and OpenClaw.
It is written so re-running it is usually safe in the same Colab session.

If Colab asks for a runtime restart, restart the runtime and run the notebook again from the top.


In [ ]:
# 3) Install Node.js and OpenClaw

# 1. Update Node.js (using the n module)
!npm install -g n
!n lts
# 2. Install OpenClaw using the official script
!curl -fsSL https://openclaw.ai/install.sh | bash


## Step 3. Verify runtime configuration

Before running the gateway, confirm that the required environment variables exist
and that the default provider/model are what you expect.


In [ ]:
# 4) Verify runtime configuration before starting OpenClaw

required_envs = [
    "OPENAI_API_KEY",
    "TELEGRAM_BOT_TOKEN",
    "ALLOWED_USER_IDS",
    "OPENCLAW_DEFAULT_MODEL",
    "OPENCLAW_MODEL_PROVIDER",
]

missing = [k for k in required_envs if not os.environ.get(k)]
if missing:
    raise RuntimeError(f"Missing required environment variables: {missing}")

print("OpenClaw is ready to start.")
print(f"Provider: {os.environ['OPENCLAW_MODEL_PROVIDER']}")
print(f"Model:    {os.environ['OPENCLAW_DEFAULT_MODEL']}")
print(f"Allowed users configured: {len(os.environ['ALLOWED_USER_IDS'].split(','))}")


## Step 4. First gateway run (pairing-code check)

This is the **first gateway run**.  
Its main purpose in Colab is to let you confirm that the gateway starts correctly
and to capture the **Telegram pairing code** shown in the logs or Telegram flow.

### Recommended Colab workflow
1. Start the gateway
2. Wait for the pairing code
3. Copy or note the code
4. **Stop the gateway cell**
5. Run the pairing approval cell
6. Optionally restore a backup
7. Start the gateway again for actual use


In [ ]:
# 5) First OpenClaw gateway run

print("Starting OpenClaw gateway (first run for pairing code)...")
!openclaw gateway run --allow-unconfigured


## Why stop the gateway cell in Colab?

On a local machine, you can keep the gateway running in one terminal and handle
pairing or restore steps in another terminal.

In Colab, the workflow is usually clearer if you:
- run the gateway once to get the pairing code
- stop the cell
- approve pairing
- optionally restore a backup
- run the gateway again

This makes the first run a **pairing-code run** and the second run the **actual working session**.


## Step 5. Approve Telegram pairing

After the first gateway run shows a pairing code:
1. stop the gateway cell
2. paste the code below
3. run the approval cell

Example:
- If the code is `1A2B3C4D`
- set `PAIRING_CODE = "1A2B3C4D"` and run the cell


In [ ]:
# 6) Approve Telegram pairing

PAIRING_CODE = ""  # Example: "1A2B3C4D"

if not PAIRING_CODE:
    print("Enter a PAIRING_CODE and run this cell again.")
else:
    !openclaw pairing approve telegram "$PAIRING_CODE"


## Step 6. Optional backup restore

Use this step **only if you already have an OpenClaw backup archive**.

Recommended workflow:
1. list available backups
2. choose one file name manually
3. restore it
4. start the gateway again

Notes:
- The default behavior is to **skip restore**
- This notebook does **not** hardcode personal backup file names
- Do **not** commit backup archives to GitHub


In [ ]:
# 7) List backups and optionally restore one

import tarfile

available_backups = sorted(DRIVE_DATA_DIR.glob("openclaw_backup_complete_*.tar.gz"))
print("Available backups:")
for p in available_backups[-20:]:
    print("-", p.name)

# Set this only when you want to restore a backup.
# Example:
# BACKUP_FILENAME = "openclaw_backup_complete_YYYYMMDD_HHMMSS.tar.gz"
BACKUP_FILENAME = None

if BACKUP_FILENAME:
    backup_path = DRIVE_DATA_DIR / BACKUP_FILENAME
    if not backup_path.exists():
        raise FileNotFoundError(f"Backup file not found: {backup_path}")

    PROJECT_DIR.mkdir(parents=True, exist_ok=True)

    with tarfile.open(backup_path, "r:gz") as tar:
        tar.extractall(PROJECT_DIR)

    print(f"[OK] Backup restored from: {backup_path}")
else:
    print("[SKIP] Restore skipped. Set BACKUP_FILENAME if you want to restore a backup.")


## Step 7. Run the gateway again for actual use

This is the **second gateway run**.  
At this point, pairing should already be approved and any optional restore should be finished.

This second run is the session you normally keep alive while using OpenClaw from Telegram.


In [ ]:
# 8) Second OpenClaw gateway run (actual working session)

print("Starting OpenClaw gateway (working session)...")
!openclaw gateway run --allow-unconfigured


## Optional: change the model

The default provider/model are:
- Provider: `openai`
- Model: `openai/gpt-4o-mini`

Use the next cell only if you want to override the model at the CLI level.
For a public GitHub notebook, it is usually best to keep the notebook default unchanged
and use Secret values or temporary overrides for private experiments.


In [ ]:
# 9) Optionally change the OpenClaw CLI model

CLI_MODEL = ""  # Example: "openai/gpt-4o-mini" or "openai/gpt-4.1-mini"

if not CLI_MODEL:
    print("CLI_MODEL is empty, so this step is skipped.")
else:
    !openclaw models set "$CLI_MODEL"


## Optional: create a backup

It is a good idea to create a backup after an important configuration change
or after a working setup has been verified.

This saves workspace, identity, and key configuration files into your Drive backup folder.


In [ ]:
# 10) Create a backup archive

%%bash
set -e

PROJECT_DIR="/root/.openclaw"
DRIVE_BACKUP_DIR="${OPENCLAW_DATA_DIR:-/content/drive/MyDrive/openclaw_data}"
TS=$(date +%Y%m%d_%H%M%S)
BACKUP_NAME="openclaw_backup_complete_${TS}.tar.gz"

mkdir -p "$DRIVE_BACKUP_DIR"

if [ ! -d "$PROJECT_DIR" ]; then
  echo "[ERROR] PROJECT_DIR not found: $PROJECT_DIR"
  exit 1
fi

cd "$PROJECT_DIR"

INCLUDE_ITEMS="workspace .env openclaw.json identity"
if [ -e "agents" ]; then
  INCLUDE_ITEMS="$INCLUDE_ITEMS agents"
fi

tar -czf "$BACKUP_NAME" --exclude='agents/main/sessions/*.lock' $INCLUDE_ITEMS

cp -f "$BACKUP_NAME" "$DRIVE_BACKUP_DIR/"
rm -f "$BACKUP_NAME"

echo "[OK] Backup created: $DRIVE_BACKUP_DIR/$BACKUP_NAME"


## Operating tips

- Always run the notebook from top to bottom
- Keep secrets only in Colab Secrets
- Do not commit backup archives or runtime state to GitHub
- In Colab, the cleanest pairing flow is:
  1. first gateway run
  2. capture pairing code
  3. stop the cell
  4. approve pairing
  5. optionally restore backup
  6. second gateway run


## Final run summary

1. Load Secrets
2. Mount Drive
3. Install Node.js and OpenClaw
4. Verify the runtime configuration
5. Run the gateway once and capture the pairing code
6. Stop the gateway cell
7. Approve Telegram pairing
8. Optionally restore a backup
9. Run the gateway again for actual use
10. Optionally change the model or create a backup

This version is intended to be safe to upload to GitHub without exposing sensitive values.
